# 🌿 EcoQuest Group Recommender — ML Model

This notebook trains and evaluates a full ML pipeline to recommend compatible groups of people for EcoQuest nature outings.

**Pipeline overview:**
1. Synthetic data generation (simulates real survey responses)
2. Feature encoding & preprocessing
3. Model training — KMeans clustering + compatibility scoring
4. Evaluation metrics
5. Group recommendation engine
6. New user prediction

## 1. Install & Import

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterGrid
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('✅ Libraries loaded')

## 2. Survey Schema & Synthetic Data Generation

In [ ]:
QUESTIONS = [
    {'key': 'age',              'options': ['18-24', '25-34', '35-44', '45+']},
    {'key': 'gender',           'options': ['Male', 'Female', 'Non-binary', 'Prefer not to say']},
    {'key': 'activity_level',   'options': ['Rarely', 'Sometimes', 'Often', 'Every day']},
    {'key': 'preferred_time',   'options': ['Morning', 'Afternoon', 'Evening', 'Flexible']},
    {'key': 'nature_interest',  'options': ['Birds', 'Insects', 'Marine life', 'Plants']},
    {'key': 'transport_mode',   'options': ['Walk', 'Bike', 'Public transit', 'Car']},
    {'key': 'quest_style',      'options': ['Solo explorer', 'Small group', 'Big group', 'Flexible']},
    {'key': 'travel_distance',  'options': ['<1 mile', '1-5 miles', '5-10 miles', '10+ miles']},
    {'key': 'experience_level', 'options': ['Beginner', 'Some experience', 'Experienced', 'Expert']},
    {'key': 'motivation',       'options': ['Learning', 'Adventure', 'Conservation', 'Social']},
]

KEYS = [q['key'] for q in QUESTIONS]

# Persona archetypes — weighted distributions that simulate real-world clusters
PERSONAS = [
    # (name, weight per question option index)  — len matches each question's options
    {'name': 'Weekend Birder',      'weights': [[1,3,2,1],[1,2,0,0],[1,2,3,1],[3,1,0,2],[4,0,0,1],[2,1,1,1],[1,3,0,2],[1,3,2,1],[1,2,3,0],[3,0,2,0]]},
    {'name': 'Marine Explorer',     'weights': [[2,3,2,1],[1,2,0,0],[0,1,3,3],[2,2,1,2],[0,0,4,0],[0,1,1,4],[0,2,1,3],[0,1,3,4],[1,2,3,2],[1,3,2,1]]},
    {'name': 'Urban Naturalist',    'weights': [[3,2,1,1],[1,2,1,1],[1,3,2,0],[1,3,2,2],[1,2,0,3],[3,2,2,0],[1,3,1,2],[3,2,1,0],[2,3,1,0],[4,0,1,2]]},
    {'name': 'Conservation Hero',   'weights': [[0,1,2,3],[1,2,1,1],[0,1,2,3],[3,1,0,2],[1,1,0,3],[0,1,2,3],[0,2,0,3],[0,1,2,3],[0,1,2,3],[0,0,4,1]]},
    {'name': 'Social Adventurer',   'weights': [[2,3,1,0],[1,2,1,0],[0,1,2,2],[1,2,2,2],[1,1,1,1],[0,2,1,3],[0,1,3,2],[0,1,3,3],[2,2,1,0],[1,2,0,4]]},
]

def generate_user(persona_idx=None):
    if persona_idx is None:
        persona_idx = np.random.randint(len(PERSONAS))
    persona = PERSONAS[persona_idx]
    row = {'persona': persona['name']}
    for qi, q in enumerate(QUESTIONS):
        w = np.array(persona['weights'][qi], dtype=float)
        # Add noise so clusters aren't perfect
        w = np.clip(w + np.random.uniform(-0.5, 0.5, len(w)), 0.1, None)
        row[q['key']] = np.random.choice(q['options'], p=w/w.sum())
    return row

N = 300
rows = [generate_user(i % len(PERSONAS)) for i in range(N)]
df = pd.DataFrame(rows)

print(f'Generated {N} synthetic users across {len(PERSONAS)} persona types')
print(f'\nDataset shape: {df.shape}')
df.head(10)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
fig.suptitle('Survey Response Distributions', fontsize=14, fontweight='bold', y=1.02)

colors = ['#1D9E75','#378ADD','#D85A30','#7F77DD','#BA7517','#D4537E','#639922','#E24B4A','#5DCAA5','#FA9921']

for idx, (q, ax) in enumerate(zip(QUESTIONS, axes.flat)):
    counts = df[q['key']].value_counts().reindex(q['options'], fill_value=0)
    bars = ax.barh(counts.index, counts.values, color=colors[idx], alpha=0.85)
    ax.set_title(q['key'].replace('_',' ').title(), fontsize=10, fontweight='bold')
    ax.set_xlabel('Count', fontsize=8)
    ax.tick_params(labelsize=8)
    for bar, val in zip(bars, counts.values):
        ax.text(val + 1, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=8)

plt.tight_layout()
plt.show()

print('\nPersona distribution:')
print(df['persona'].value_counts())

## 4. Feature Engineering & Encoding

In [ ]:
# Ordinal encoding — preserves natural order in ordered features
feature_cols = KEYS
X_raw = df[feature_cols].copy()

categories = [q['options'] for q in QUESTIONS]
encoder = OrdinalEncoder(categories=categories)
X_encoded = encoder.fit_transform(X_raw)

# Normalize to 0-1 range so all features have equal weight
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

print(f'Feature matrix shape: {X_scaled.shape}')
print(f'\nEncoded sample (first 3 rows):')
pd.DataFrame(X_encoded[:3], columns=feature_cols).round(2)

## 5. Find Optimal Number of Clusters (Elbow + Silhouette)

In [ ]:
K_range = range(2, 12)
inertias, silhouettes, db_scores = [], [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Cluster Evaluation Metrics', fontsize=13, fontweight='bold')

axes[0].plot(K_range, inertias, 'o-', color='#1D9E75', linewidth=2)
axes[0].set_xlabel('Number of clusters (k)'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method'); axes[0].grid(alpha=0.3)

best_k_sil = list(K_range)[np.argmax(silhouettes)]
axes[1].plot(K_range, silhouettes, 'o-', color='#378ADD', linewidth=2)
axes[1].axvline(best_k_sil, color='red', linestyle='--', alpha=0.5, label=f'Best k={best_k_sil}')
axes[1].set_xlabel('Number of clusters (k)'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (higher = better)'); axes[1].legend(); axes[1].grid(alpha=0.3)

best_k_db = list(K_range)[np.argmin(db_scores)]
axes[2].plot(K_range, db_scores, 'o-', color='#D85A30', linewidth=2)
axes[2].axvline(best_k_db, color='red', linestyle='--', alpha=0.5, label=f'Best k={best_k_db}')
axes[2].set_xlabel('Number of clusters (k)'); axes[2].set_ylabel('Davies-Bouldin Score')
axes[2].set_title('Davies-Bouldin Score (lower = better)'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

BEST_K = best_k_sil
print(f'\n✅ Selected k = {BEST_K} (best silhouette score: {max(silhouettes):.3f})')

## 6. Train KMeans Model

In [ ]:
kmeans = KMeans(n_clusters=BEST_K, random_state=42, n_init=20, max_iter=500)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Evaluate
sil = silhouette_score(X_scaled, df['cluster'])
db  = davies_bouldin_score(X_scaled, df['cluster'])

print(f'KMeans (k={BEST_K}) — Final Model')
print(f'  Silhouette Score : {sil:.4f}  (closer to 1 = better)')
print(f'  Davies-Bouldin   : {db:.4f}  (closer to 0 = better)')
print(f'\nCluster sizes:')
print(df['cluster'].value_counts().sort_index())

print('\nCluster vs Persona cross-tab:')
pd.crosstab(df['cluster'], df['persona'])

## 7. Cluster Visualization (PCA 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)
centers_2d = pca.transform(kmeans.cluster_centers_)

palette = ['#1D9E75','#378ADD','#D85A30','#7F77DD','#BA7517','#D4537E','#639922','#E24B4A','#5DCAA5','#FA9921']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By cluster
for c in sorted(df['cluster'].unique()):
    mask = df['cluster'] == c
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=palette[c % len(palette)],
                    label=f'Cluster {c}', alpha=0.6, s=40)
axes[0].scatter(centers_2d[:, 0], centers_2d[:, 1], c='black', marker='X', s=120, zorder=5, label='Centroids')
axes[0].set_title('KMeans Clusters (PCA)', fontweight='bold')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.2)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')

# By persona
personas = df['persona'].unique()
for i, p in enumerate(personas):
    mask = df['persona'] == p
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], c=palette[i % len(palette)],
                    label=p, alpha=0.6, s=40)
axes[1].set_title('True Personas (PCA)', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.2)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')

plt.suptitle('Cluster vs Persona Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%')

## 8. Cluster Profiling — What does each cluster look like?

In [ ]:
def profile_cluster(df, cluster_id):
    subset = df[df['cluster'] == cluster_id]
    print(f"\n{'='*50}")
    print(f"Cluster {cluster_id}  (n={len(subset)})")
    print('='*50)
    for q in QUESTIONS:
        top = subset[q['key']].value_counts().index[0]
        pct = subset[q['key']].value_counts().iloc[0] / len(subset) * 100
        print(f"  {q['key']:20s}: {top:20s} ({pct:.0f}%)")

for c in sorted(df['cluster'].unique()):
    profile_cluster(df, c)

## 9. Compatibility Scoring Engine

In [ ]:
# Feature weights — some dimensions matter more for quest compatibility
FEATURE_WEIGHTS = {
    'age':              0.5,   # minor — age diversity is fine
    'gender':           0.3,   # not very relevant
    'activity_level':   1.5,   # important — must match pace
    'preferred_time':   2.0,   # critical — scheduling
    'nature_interest':  1.8,   # important — shared focus
    'transport_mode':   1.2,   # relevant — getting there together
    'quest_style':      1.5,   # important — group dynamics
    'travel_distance':  1.5,   # important — range overlap
    'experience_level': 1.0,   # moderate — mixed is ok
    'motivation':       1.2,   # relevant — why they're there
}

weight_vec = np.array([FEATURE_WEIGHTS[k] for k in KEYS])

def compute_similarity(vec_a, vec_b, weights=weight_vec):
    """Weighted cosine similarity between two encoded user vectors."""
    a = vec_a * weights
    b = vec_b * weights
    dot = np.dot(a, b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(dot / denom) if denom > 0 else 0.0

def group_compatibility_score(indices, X):
    """Average pairwise weighted cosine similarity for a group."""
    pairs = list(combinations(indices, 2))
    if not pairs:
        return 0.0
    scores = [compute_similarity(X[i], X[j]) for i, j in pairs]
    return np.mean(scores)

# Precompute full similarity matrix
X_weighted = X_encoded * weight_vec
sim_matrix = cosine_similarity(X_weighted)

print('Compatibility scoring engine ready.')
print(f'Similarity matrix shape: {sim_matrix.shape}')
print(f'\nSample pair similarity (user 0 vs user 1): {sim_matrix[0,1]:.3f}')

## 10. Group Recommendation — Find Best Groups

In [ ]:
def recommend_groups(df, X_encoded, n_top=5, group_size=4, same_cluster_bonus=0.05):
    """
    Finds the top-N most compatible groups of `group_size` people.
    Applies a bonus if all members share a cluster (similar persona).
    """
    n = len(df)
    names = df['name'].tolist() if 'name' in df.columns else [f'User_{i}' for i in range(n)]
    clusters = df['cluster'].tolist()

    results = []
    for combo in combinations(range(n), group_size):
        score = group_compatibility_score(list(combo), X_encoded)
        # Same-cluster bonus
        if len(set(clusters[i] for i in combo)) == 1:
            score += same_cluster_bonus
        results.append({'members': [names[i] for i in combo],
                        'indices': list(combo),
                        'score': score,
                        'clusters': [clusters[i] for i in combo]})

    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:n_top]

# Demo with first 20 users
df_demo = df.head(20).copy().reset_index(drop=True)
X_demo  = X_encoded[:20]

top_groups = recommend_groups(df_demo, X_demo, n_top=5, group_size=4)

print('Top 5 recommended EcoQuest groups (from 20 demo users)\n')
print(f'{"Rank":<6}{"Score":>8}   {"Members":<50} {"Clusters"}')
print('-'*90)
for rank, g in enumerate(top_groups, 1):
    members_str = ', '.join(g['members'])
    clusters_str = str(g['clusters'])
    print(f'{rank:<6}{g["score"]:>8.4f}   {members_str:<50} {clusters_str}')

## 11. Visualize: Similarity Heatmap for Demo Users

In [ ]:
sim_demo = cosine_similarity(X_demo * weight_vec)
labels_demo = [f'U{i} C{df_demo["cluster"].iloc[i]}' for i in range(20)]

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.eye(len(sim_demo), dtype=bool)
sns.heatmap(sim_demo, annot=True, fmt='.2f', cmap='YlGn',
            xticklabels=labels_demo, yticklabels=labels_demo,
            mask=mask, linewidths=0.3, ax=ax, vmin=0.5, vmax=1.0,
            annot_kws={'size': 8})
ax.set_title('Pairwise Compatibility Matrix (demo users)', fontweight='bold', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 12. Recommend for a New User (Inference)

In [ ]:
def encode_user(answers: dict) -> np.ndarray:
    """Encode a single user's survey answers into a feature vector."""
    row = []
    for q in QUESTIONS:
        val = answers.get(q['key'], q['options'][0])
        idx = q['options'].index(val) if val in q['options'] else 0
        row.append(float(idx))
    return np.array(row)

def predict_cluster(answers: dict) -> int:
    """Assign a new user to the nearest cluster."""
    vec = encode_user(answers)
    vec_scaled = scaler.transform([vec])
    return int(kmeans.predict(vec_scaled)[0])

def find_best_partners(new_user_answers: dict, df_pool: pd.DataFrame,
                        X_pool: np.ndarray, group_size=3, top_n=5):
    """
    Given a new user's answers and a pool of existing users,
    return the best (group_size - 1) companions to form the top groups.
    """
    new_vec = encode_user(new_user_answers)
    new_cluster = predict_cluster(new_user_answers)

    names = df_pool['name'].tolist() if 'name' in df_pool.columns else [f'User_{i}' for i in range(len(df_pool))]

    # Score new user against each pool member
    pairwise = [(i, compute_similarity(new_vec, X_pool[i])) for i in range(len(df_pool))]
    pairwise.sort(key=lambda x: x[1], reverse=True)

    # Now find best (group_size-1) companions
    companion_pool = [p[0] for p in pairwise[:20]]  # top-20 candidates
    best_groups = []
    for combo in combinations(companion_pool, group_size - 1):
        all_idx = list(combo)
        all_vecs = np.vstack([new_vec] + [X_pool[i] for i in all_idx])
        score = group_compatibility_score(list(range(len(all_vecs))), all_vecs)
        best_groups.append({'companions': [names[i] for i in all_idx], 'score': score})

    best_groups.sort(key=lambda x: x['score'], reverse=True)
    return new_cluster, best_groups[:top_n]


# --- Example: new user survey ---
new_user = {
    'age':              '25-34',
    'gender':           'Female',
    'activity_level':   'Often',
    'preferred_time':   'Morning',
    'nature_interest':  'Birds',
    'transport_mode':   'Bike',
    'quest_style':      'Small group',
    'travel_distance':  '5-10 miles',
    'experience_level': 'Experienced',
    'motivation':       'Conservation',
}

assigned_cluster, recommendations = find_best_partners(
    new_user, df_demo, X_demo, group_size=4, top_n=5
)

print(f'New user assigned to Cluster {assigned_cluster}\n')
print('Top recommended quest groups for new user:\n')
for rank, g in enumerate(recommendations, 1):
    print(f'  {rank}. Companions: {g["companions"]}  — Score: {g["score"]:.4f}')

## 13. Feature Importance via Cluster Centroid Variance

In [ ]:
# How much does each feature vary across cluster centroids?
# High variance = feature strongly drives cluster separation
centers = scaler.inverse_transform(kmeans.cluster_centers_)  # back to encoded space
centroid_variance = np.var(centers, axis=0)

importance_df = pd.DataFrame({
    'feature': KEYS,
    'centroid_variance': centroid_variance,
    'weight': [FEATURE_WEIGHTS[k] for k in KEYS]
}).sort_values('centroid_variance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(importance_df['feature'], importance_df['centroid_variance'],
               color='#1D9E75', alpha=0.85)
ax.set_xlabel('Centroid Variance (feature importance proxy)', fontsize=10)
ax.set_title('Feature Importance for Cluster Separation', fontweight='bold', fontsize=12)
for bar, val in zip(bars, importance_df['centroid_variance']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Save Model Artifacts

In [ ]:
import pickle, os

artifacts = {
    'kmeans':     kmeans,
    'encoder':    encoder,
    'scaler':     scaler,
    'pca':        pca,
    'questions':  QUESTIONS,
    'weights':    FEATURE_WEIGHTS,
    'best_k':     BEST_K,
    'sil_score':  sil,
    'db_score':   db,
}

os.makedirs('ecoquest_model', exist_ok=True)
with open('ecoquest_model/model.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

df.to_csv('ecoquest_model/training_data.csv', index=False)

print('✅ Model saved to ecoquest_model/model.pkl')
print('✅ Training data saved to ecoquest_model/training_data.csv')
print(f'\nFinal model summary:')
print(f'  Algorithm    : KMeans + Weighted Cosine Similarity')
print(f'  Clusters (k) : {BEST_K}')
print(f'  Silhouette   : {sil:.4f}')
print(f'  Davies-Bouldin: {db:.4f}')
print(f'  Training size: {len(df)} users')

## 15. Load Model & Quick Predict (Production Usage)

In [ ]:
# Load saved model
with open('ecoquest_model/model.pkl', 'rb') as f:
    model = pickle.load(f)

def quick_recommend(user_answers: dict, pool_df: pd.DataFrame, pool_X: np.ndarray,
                     group_size=4):
    """One-call wrapper for production inference."""
    vec = encode_user(user_answers)
    vec_s = model['scaler'].transform([vec])
    cluster = int(model['kmeans'].predict(vec_s)[0])

    _, groups = find_best_partners(user_answers, pool_df, pool_X, group_size=group_size)
    return cluster, groups

# Run
cluster, groups = quick_recommend(new_user, df_demo, X_demo)
print(f'Cluster: {cluster}')
print(f'Best group: {groups[0]["companions"]} (score={groups[0]["score"]:.4f})')